# Module 03 — Exercises

Exercises for **Module 03 — Eigenvalues, SVD and PCA**. They follow the order of the
[concept guide](README.md): eigenvalues by hand, the spectral theorem, truncated SVD, power
iteration and PageRank, PCA, the pseudo-inverse, conditioning, ridge regression through the
SVD, two conceptual questions and a mini-project on latent-factor recommendation.
Use only NumPy (plus the scikit-learn objects named in a prompt) and keep everything seeded.

**How these exercises work.** Each coding exercise has a *starter* cell (replace `# YOUR CODE HERE` with your implementation) followed by a *check* cell. Run the check cell to get an instant ✅/❌. Checks never stop the notebook, so you can run everything top-to-bottom at any time and see your score at the end. Some checks compare against a *hidden* answer (only a hash is stored), so you cannot peek at the number — think first, then verify. Conceptual questions hide their answer behind a *Show answer* toggle. Mini-projects are open-ended and come with a rubric. Difficulty is marked ★☆☆ (warm-up), ★★☆ (core), ★★★ (stretch).

### Setup and self-check helpers
Run the next two cells first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_wine
from sklearn.decomposition import PCA
rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (7, 4)
np.set_printoptions(precision=4, suppress=True)

In [ ]:
import hashlib as _hashlib
import numpy as _np

_results = {}

def _canon(value, decimals=3):
    """Canonical string form of an answer, so hashes are stable."""
    if isinstance(value, (bool, _np.bool_)):
        return "true" if value else "false"
    if isinstance(value, str):
        return " ".join(value.strip().lower().split())
    if isinstance(value, (int, _np.integer)):
        return str(int(value))
    if isinstance(value, (float, _np.floating)):
        s = f"{float(value):.{decimals}f}"
        return "0." + "0" * decimals if s in ("-0." + "0" * decimals, "-0") else s
    if isinstance(value, dict):
        return "{" + ",".join(f"{_canon(k, decimals)}:{_canon(v, decimals)}" for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))) + "}"
    try:
        arr = _np.asarray(value)
        if arr.ndim == 0:
            return _canon(arr.item(), decimals)
        return "[" + ",".join(_canon(x, decimals) for x in arr.ravel().tolist()) + "]"
    except Exception:
        return str(value)

def sha_of(value, decimals=3):
    return _hashlib.sha256(_canon(value, decimals).encode()).hexdigest()[:16]

def check(name, condition, hint=""):
    """Record and print a pass/fail mark. Never raises, so the notebook keeps running."""
    try:
        ok = bool(condition)
    except Exception:
        ok = False
    _results[name] = ok
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"   — hint: {hint}"))
    return ok

def check_close(name, got, expected, tol=1e-6, hint=""):
    try:
        g = _np.asarray(got, dtype=float)
        e = _np.asarray(expected, dtype=float)
        ok = g.shape == e.shape and bool(_np.allclose(g, e, atol=tol, rtol=tol))
    except Exception:
        ok = False
    return check(name, ok, hint)

def check_hash(name, value, expected_sha, decimals=3, hint=""):
    """Compare an answer against a hidden expected value (only its hash is stored)."""
    try:
        ok = value is not None and sha_of(value, decimals) == expected_sha
    except Exception:
        ok = False
    return check(name, ok, hint)

def summary():
    n_ok = sum(1 for v in _results.values() if v)
    n = len(_results)
    print(f"\nScore: {n_ok}/{n} checks passed" + ("  🎉" if n and n_ok == n else ""))
    if n and n_ok < n:
        print("Not yet passing: " + ", ".join(k for k, v in _results.items() if not v))

## Exercise 1 — Eigenvalues of a 2x2 matrix by hand  `★☆☆`

Implement `eig2x2(A)` that returns the two eigenvalues of a real $2\times2$ matrix `A`
as a NumPy array **sorted in descending order**, using only the characteristic polynomial

$$\lambda^2 - \operatorname{tr}(A)\,\lambda + \det(A) = 0$$

and the quadratic formula. Do **not** call anything from `np.linalg`. You may assume the
eigenvalues are real (discriminant $\ge 0$).

In [ ]:
def eig2x2(A):
    # YOUR CODE HERE  (trace, determinant, quadratic formula; return np.array([larger, smaller]))
    return None

**Check your answer:**

In [ ]:
try:
    got = eig2x2(np.array([[4., 1.], [2., 3.]]))
except Exception:
    got = None
check_close('eigenvalues of [[4,1],[2,3]] are 5 and 2', got, np.array([5., 2.]))
try:
    got_sym = eig2x2(np.array([[2., 1.], [1., 2.]]))
except Exception:
    got_sym = None
check_close('eigenvalues of [[2,1],[1,2]] are 3 and 1', got_sym, np.array([3., 1.]))
try:
    got_hidden = eig2x2(np.array([[1., 4.], [2., 3.]]))
except Exception:
    got_hidden = None
check_hash('hidden: eigenvalues of [[1,4],[2,3]]', got_hidden, '45f8a93c2836fbf1')

## Exercise 2 — Spectral theorem and positive semidefiniteness  `★☆☆`

Implement two functions for a real **symmetric** matrix `A`:

* `spectral(A)` returns `(lam, Q)` where `lam` holds the eigenvalues **sorted in descending
  order** and the columns of `Q` are the matching **orthonormal** eigenvectors, so that
  `Q @ np.diag(lam) @ Q.T` reconstructs `A`. Use `np.linalg.eigh` (which returns ascending order).
* `is_psd(A, tol=1e-10)` returns a Python `bool`: `True` if every eigenvalue is $\ge -\text{tol}$.

In [ ]:
def spectral(A):
    # YOUR CODE HERE  (np.linalg.eigh, then reorder to descending)
    return None

def is_psd(A, tol=1e-10):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
_g = np.random.default_rng(11)
_B = _g.normal(size=(4, 4)); _S = _B @ _B.T
try:
    lam_s, Q_s = spectral(_S)
except Exception:
    lam_s, Q_s = None, None
check('eigenvalues sorted in descending order', lam_s is not None and np.all(np.diff(lam_s) <= 0))
check('Q is orthonormal (Q^T Q = I)', Q_s is not None and np.allclose(Q_s.T @ Q_s, np.eye(4)))
check('Q diag(lam) Q^T reconstructs A', Q_s is not None and np.allclose(Q_s @ np.diag(lam_s) @ Q_s.T, _S))
try:
    psd_a = is_psd(np.array([[2., 1.], [1., 2.]]))
    psd_b = is_psd(np.array([[1., 2.], [2., 1.]]))
    psd_c = is_psd(_S)
except Exception:
    psd_a = psd_b = psd_c = None
check('[[2,1],[1,2]] is PSD', psd_a is True)
check('[[1,2],[2,1]] is NOT PSD', psd_b is False)
check('a Gram matrix B B^T is PSD', psd_c is True)
try:
    smallest = float(spectral(np.array([[1., 2.], [2., 1.]]))[0].min())
except Exception:
    smallest = None
check_hash('hidden: smallest eigenvalue of [[1,2],[2,1]]', smallest, '9b225070ea9ac4db')

## Exercise 3 — Truncated SVD and the Eckart-Young error  `★☆☆`

Implement

* `rank_k(A, k)` — the best rank-$k$ approximation $A_k = U_k\Sigma_kV_k^\top$ built from
  `np.linalg.svd(A, full_matrices=False)`;
* `rel_frobenius_error(A, k)` — the relative error $\|A - A_k\|_F / \|A\|_F$ as a Python `float`.

Eckart–Young says this error equals $\sqrt{\sum_{i>k}\sigma_i^2 / \sum_i \sigma_i^2}$; the
check compares your measured error with that formula.

In [ ]:
def rank_k(A, k):
    # YOUR CODE HERE
    return None

def rel_frobenius_error(A, k):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
_g = np.random.default_rng(7)
_M = _g.normal(size=(20, 3)) @ _g.normal(size=(3, 15)) + 0.5 * _g.normal(size=(20, 15))
_s = np.linalg.svd(_M, compute_uv=False)
try:
    _Ak = rank_k(_M, 2)
except Exception:
    _Ak = None
check('rank_k(M, 2) has shape (20, 15)', _Ak is not None and _Ak.shape == (20, 15))
check('rank_k(M, 2) has rank 2', _Ak is not None and np.linalg.matrix_rank(_Ak) == 2)
try:
    _full = rank_k(_M, 15)
except Exception:
    _full = None
check_close('rank_k(M, 15) reproduces M exactly', _full, _M, tol=1e-8)
try:
    _e2 = rel_frobenius_error(_M, 2)
except Exception:
    _e2 = None
check_close('k = 2 error matches sqrt(sum_{i>2} s_i^2 / sum s_i^2)', _e2,
            np.sqrt((_s[2:] ** 2).sum() / (_s ** 2).sum()), tol=1e-8)
try:
    _e3 = rel_frobenius_error(_M, 3)
except Exception:
    _e3 = None
check_hash('hidden: relative error at k = 3 (3 decimals)', _e3, '79308bed382bc45a')

## Exercise 4 — Power iteration  `★★☆`

Implement `power_iteration(A, n_iter=500, x0=None)` returning `(lam, v)`: the dominant
eigenvalue (as a Python `float`, estimated by the Rayleigh quotient $v^\top A v$) and the
corresponding **unit-length** eigenvector. Start from `x0` if given, otherwise from the
all-ones vector; normalise after every multiplication. Do **not** call `np.linalg.eig`,
`eigh`, `eigvals` or `svd`.

In [ ]:
def power_iteration(A, n_iter=500, x0=None):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
_g = np.random.default_rng(18)
_B = _g.normal(size=(5, 5)); _S5 = _B @ _B.T
_lam_true, _Q_true = np.linalg.eigh(_S5)
try:
    lam_pi, v_pi = power_iteration(_S5)
except Exception:
    lam_pi, v_pi = None, None
check_close('dominant eigenvalue of a 5x5 Gram matrix', lam_pi, _lam_true[-1], tol=1e-8)
check('eigenvector has unit length', v_pi is not None and abs(np.linalg.norm(v_pi) - 1) < 1e-9)
check('eigenvector matches eigh up to sign', v_pi is not None and abs(abs(v_pi @ _Q_true[:, -1]) - 1) < 1e-6)
try:
    lam_A, v_A = power_iteration(np.array([[4., 1.], [2., 3.]]), x0=np.array([1., 0.]))
except Exception:
    lam_A, v_A = None, None
check_close('works on the non-symmetric [[4,1],[2,3]]: lambda_1 = 5', lam_A, 5.0, tol=1e-8)
check_hash('hidden: dominant eigenvalue of the Gram matrix (3 decimals)', lam_pi, '99c7bf1b88cd3f33')

## Exercise 5 — PageRank with damping  `★★☆`

Implement `pagerank(links, d=0.85, n_iter=200)` for a web graph given as a dictionary
`links[j] = list of pages that page j links to` (pages are numbered `0 .. n-1`). Build the
column-stochastic matrix $M$ with $M_{ij} = 1/\text{outdeg}(j)$ if $j$ links to $i$, treating a
page with **no out-links as linking to every page** (including itself) with probability $1/n$.
Form the Google matrix

$$G = d\,M + \frac{1-d}{n}\,\mathbf{1}\mathbf{1}^\top ,$$

start from the uniform vector and apply $G$ `n_iter` times. Return the PageRank vector
(a NumPy array that sums to one).

The hidden checks use this seven-page web (page 6 is a dead end):

```python
web = {0: [1, 3], 1: [2], 2: [0, 1], 3: [2, 4], 4: [5], 5: [3, 4], 6: []}
```

In [ ]:
def pagerank(links, d=0.85, n_iter=200):
    n = len(links)
    # YOUR CODE HERE  (build M column by column, then G, then iterate)
    return None

web = {0: [1, 3], 1: [2], 2: [0, 1], 3: [2, 4], 4: [5], 5: [3, 4], 6: []}

**Check your answer:**

In [ ]:
try:
    pr_two = pagerank({0: [1], 1: [0]})
except Exception:
    pr_two = None
check_close('two pages linking to each other share rank equally', pr_two, np.array([0.5, 0.5]), tol=1e-9)
try:
    pr = pagerank(web)
except Exception:
    pr = None
check('PageRank sums to one', pr is not None and abs(pr.sum() - 1) < 1e-9)
check('every page has positive rank (damping)', pr is not None and np.all(pr > 0))
check('the dead-end page 6 gets the lowest rank', pr is not None and int(np.argmin(pr)) == 6)
check_hash('hidden: the top-ranked page', int(np.argmax(pr)) if pr is not None else None, 'd4735e3a265e16ee')
check_hash('hidden: the full ranking, best first', [int(i) for i in np.argsort(-pr)] if pr is not None else None, 'f3ef42c421cc6ed5')

## Exercise 6 — PCA from scratch versus scikit-learn  `★★☆`

Implement PCA with NumPy only:

* `pca_fit(X, k)` returns `(components, explained_variance_ratio, mean)` where `components`
  is a $k\times d$ array whose **rows** are the first $k$ principal components (unit length),
  `explained_variance_ratio` has shape `(k,)`, and `mean` is the column mean of `X`. Centre
  the data, then use either the eigendecomposition of the covariance matrix (with $n-1$) or
  the SVD of the centred data.
* `pca_transform(X, components, mean)` returns the $n\times k$ matrix of scores.

Then standardise the wine data `load_wine().data` (zero mean, unit variance per column),
run your PCA with all 13 components, and set `k95` to the **smallest** number of components
whose cumulative explained variance reaches 95 % (a Python `int`).

In [ ]:
def pca_fit(X, k):
    # YOUR CODE HERE
    return None

def pca_transform(X, components, mean):
    # YOUR CODE HERE
    return None

wine = load_wine().data
k95 = None   # YOUR CODE HERE

**Check your answer:**

In [ ]:
_X = load_iris().data
_ref = PCA(n_components=2).fit(_X)
try:
    comps, evr, mu_ = pca_fit(_X, 2)
except Exception:
    comps, evr, mu_ = None, None, None
check('components has shape (2, 4)', comps is not None and comps.shape == (2, 4))
check_close('explained variance ratio matches sklearn', evr, _ref.explained_variance_ratio_, tol=1e-8)
check('components match sklearn up to sign', comps is not None and np.allclose(np.abs(comps), np.abs(_ref.components_), atol=1e-6))
try:
    Z = pca_transform(_X, comps, mu_)
except Exception:
    Z = None
check('scores have shape (150, 2)', Z is not None and Z.shape == (150, 2))
check('scores match sklearn up to sign', Z is not None and np.allclose(np.abs(Z), np.abs(_ref.transform(_X)), atol=1e-6))
_lam_all = np.linalg.eigvalsh(np.cov(_X, rowvar=False))[::-1]
try:
    _resid = ((_X - (Z @ comps + mu_)) ** 2).sum()
except Exception:
    _resid = None
check_close('reconstruction error equals (n-1) * sum of dropped eigenvalues', _resid, 149 * _lam_all[2:].sum(), tol=1e-6)
check_hash('hidden: components for 95% of variance (standardised wine)', k95 if isinstance(k95, (int, np.integer)) else None, '4a44dc15364204a8')

## Exercise 7 — Why PCA on unscaled features is misleading  `★★☆`

A hospital dataset has the columns *age (years)*, *systolic blood pressure (mmHg)*,
*cholesterol (mg/dL)* and *daily steps (count)*. A colleague runs PCA on the raw data and
reports that "the first component explains 99.8 % of the variance, so the data is essentially
one-dimensional".

1. Explain what the first component almost certainly is and why the 99.8 % figure says
   nothing about the structure of the data.
2. State precisely what preprocessing fixes the problem and what PCA then operates on.
3. Name one situation in which you would deliberately **not** standardise before PCA.

<details><summary><b>Show answer</b></summary>

1. Daily step counts vary by thousands, while age varies by tens and blood pressure and
   cholesterol by tens to hundreds. The total variance is therefore dominated by the steps
   column, and the first principal component is essentially the steps axis (a loading near 1
   on steps and near 0 elsewhere). Its 99.8 % is a statement about *units*, not about the
   data: re-recording steps in thousands of steps would change the result completely. PCA
   maximises variance, and variance is not scale-invariant.
2. Standardise every column — subtract its mean and divide by its standard deviation — so
   that each feature has unit variance. PCA then diagonalises the **correlation matrix**
   instead of the covariance matrix; each feature starts with an equal share (1 of $d$) of the
   variance, and the components reflect correlations between features rather than their
   magnitudes.
3. When the features share a unit and their scale *is* meaningful: pixel intensities in an
   image, gene-expression values after a common log transform, or returns of several assets in
   percent. Standardising would then inflate the noise in nearly constant features (a dead
   pixel, a gene that is barely expressed) to the same importance as real signal. In such
   cases run PCA on the covariance matrix, possibly after a variance-stabilising transform.

</details>

## Exercise 8 — The pseudo-inverse via the SVD  `★★☆`

Implement `pinv_svd(A, rcond=1e-12)` returning the Moore–Penrose pseudo-inverse
$A^{+} = V\Sigma^{+}U^\top$ for any real $m\times n$ matrix, where $\Sigma^{+}$ inverts every
singular value with $\sigma_i > \text{rcond}\cdot\sigma_{\max}$ and sets the rest to zero.
Use `np.linalg.svd` only — no `np.linalg.pinv`, `inv`, `solve` or `lstsq`.

Then compute `x_min = pinv_svd(A) @ b` for the under-determined system

$$A = \begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{pmatrix}, \qquad b = (1, 1),$$

which has infinitely many exact solutions; `x_min` is the one with the smallest norm.

In [ ]:
def pinv_svd(A, rcond=1e-12):
    # YOUR CODE HERE
    return None

A_under = np.array([[1., 2., 3.], [4., 5., 6.]])
b_under = np.array([1., 1.])
x_min = None   # YOUR CODE HERE  (use pinv_svd)

**Check your answer:**

In [ ]:
_g = np.random.default_rng(5)
_T = _g.normal(size=(6, 3))                               # tall, full column rank
_R = _g.normal(size=(4, 2)) @ _g.normal(size=(2, 4))      # 4x4 of rank 2
try:
    P_T = pinv_svd(_T); P_R = pinv_svd(_R)
except Exception:
    P_T = P_R = None
check_close('matches np.linalg.pinv on a 6x3 full-rank matrix', P_T, np.linalg.pinv(_T), tol=1e-8)
check_close('matches np.linalg.pinv on a rank-deficient 4x4 matrix', P_R, np.linalg.pinv(_R), tol=1e-8)
check('Moore-Penrose identity A A+ A = A (rank-deficient case)', P_R is not None and np.allclose(_R @ P_R @ _R, _R, atol=1e-8))
check('full column rank: A+ equals (A^T A)^-1 A^T', P_T is not None and np.allclose(P_T, np.linalg.solve(_T.T @ _T, _T.T), atol=1e-8))
check('x_min solves the under-determined system exactly', x_min is not None and np.allclose(A_under @ x_min, b_under, atol=1e-9))
_x_other = (x_min + 0.3 * np.array([1., -2., 1.])) if x_min is not None else None    # (1,-2,1) spans null(A)
check('any other solution x_min + z, z in null(A), is longer', _x_other is not None and np.linalg.norm(_x_other) > np.linalg.norm(x_min))
check_hash('hidden: norm of the minimum-norm solution (3 decimals)', float(np.linalg.norm(x_min)) if x_min is not None else None, '31dd550a036488a5')

## Exercise 9 — Condition number and error amplification  `★★☆`

Implement

* `cond_svd(A)` — the 2-norm condition number $\kappa(A) = \sigma_{\max}/\sigma_{\min}$ from the
  singular values (a Python `float`; do not call `np.linalg.cond`);
* `amplification(A, b, db)` — solve $Ax = b$ and $A\tilde{x} = b + \delta b$ with `np.linalg.solve`
  and return the ratio $\dfrac{\|\tilde x - x\|/\|x\|}{\|\delta b\|/\|b\|}$ as a `float`.

Then set `kappa_H4 = cond_svd(H4)` for the $4\times4$ Hilbert matrix $H_{ij} = 1/(i+j-1)$ (1-based
indices), a famous ill-conditioned matrix. The theory says `amplification` can never exceed
`cond_svd(A)` and reaches it when $b$ lies along $u_1$ and $\delta b$ along $u_{\min}$.

In [ ]:
def cond_svd(A):
    # YOUR CODE HERE
    return None

def amplification(A, b, db):
    # YOUR CODE HERE
    return None

H4 = np.array([[1.0 / (i + j + 1) for j in range(4)] for i in range(4)])   # the 4x4 Hilbert matrix
kappa_H4 = None   # YOUR CODE HERE

**Check your answer:**

In [ ]:
try:
    k_H4 = cond_svd(H4)
except Exception:
    k_H4 = None
check_close('cond_svd matches np.linalg.cond on the Hilbert matrix', k_H4, np.linalg.cond(H4), tol=1e-6)
check_hash('hidden: condition number of the 4x4 Hilbert matrix (3 decimals)', kappa_H4 if isinstance(kappa_H4, float) else None, '0e17fbbf53590fd6')
_g = np.random.default_rng(21)
_Q1, _ = np.linalg.qr(_g.normal(size=(5, 5))); _Q2, _ = np.linalg.qr(_g.normal(size=(5, 5)))
_s = np.array([1.0, 0.5, 0.1, 0.01, 1e-4])
_A = _Q1 @ np.diag(_s) @ _Q2.T                             # kappa = 1e4 by construction
try:
    amp_rand = amplification(_A, _g.normal(size=5), 1e-6 * _g.normal(size=5))
    amp_worst = amplification(_A, _Q1[:, 0], 1e-6 * _Q1[:, -1])
except Exception:
    amp_rand = amp_worst = None
check('random perturbation: amplification <= kappa', amp_rand is not None and 1.0 <= amp_rand <= 1e4 * (1 + 1e-6))
check_close('worst-case directions: amplification equals kappa = 1e4', amp_worst, 1e4, tol=1e-4)

## Exercise 10 — Ridge regression as singular-value shrinkage  `★★★`

Ridge regression minimises $\|Xw - y\|^2 + \lambda\|w\|^2$ and has the closed form
$w_\lambda = (X^\top X + \lambda I)^{-1}X^\top y$. Show that it is a *filtered* pseudo-inverse by
implementing it through the SVD $X = U\Sigma V^\top$:

$$w_\lambda = \sum_i \frac{\sigma_i}{\sigma_i^2 + \lambda}\,(u_i^\top y)\,v_i .$$

Implement `ridge_svd(X, y, lam)` using `np.linalg.svd(X, full_matrices=False)` only (no
`inv`, `solve` or `lstsq`). The checks compare against the closed form, verify that
$\lambda = 0$ recovers the least-squares solution, and that larger $\lambda$ gives a shorter
coefficient vector.

In [ ]:
def ridge_svd(X, y, lam):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
_g = np.random.default_rng(13)
_X = _g.normal(size=(40, 6)) @ (np.eye(6) + 0.8 * np.ones((6, 6)))   # correlated features
_w_true = np.array([1., -2., 0.5, 0., 3., -1.])
_y = _X @ _w_true + 0.5 * _g.normal(size=40)
def _closed_form(lam):
    return np.linalg.solve(_X.T @ _X + lam * np.eye(6), _X.T @ _y)
try:
    w1 = ridge_svd(_X, _y, 1.0); w10 = ridge_svd(_X, _y, 10.0); w0 = ridge_svd(_X, _y, 0.0)
except Exception:
    w1 = w10 = w0 = None
check_close('lambda = 1 matches (X^T X + lambda I)^-1 X^T y', w1, _closed_form(1.0), tol=1e-8)
check_close('lambda = 10 matches the closed form', w10, _closed_form(10.0), tol=1e-8)
check_close('lambda = 0 recovers ordinary least squares', w0, np.linalg.lstsq(_X, _y, rcond=None)[0], tol=1e-8)
check('coefficients shrink as lambda grows', w0 is not None and np.linalg.norm(w10) < np.linalg.norm(w1) < np.linalg.norm(w0))

## Exercise 11 — Diagnose a numerically unsafe SVD  `★★☆`

A colleague computes singular values with the recipe "form $A^\top A$, call `np.linalg.eigh`,
take square roots". It agrees with `np.linalg.svd` on random matrices, so they conclude the
two are equivalent.

Explain why the recipe is numerically unsafe in general. Use the $3\times2$ matrix

$$A = \begin{pmatrix} 1 & 1 \\ \varepsilon & 0 \\ 0 & \varepsilon \end{pmatrix}, \qquad \varepsilon = 10^{-8},$$

work out its exact singular values, and predict what the recipe returns in double precision
(about 16 significant digits). What does this have to do with the condition number?

<details><summary><b>Show answer</b></summary>

Exactly, $A^\top A = \begin{pmatrix} 1+\varepsilon^2 & 1 \\ 1 & 1+\varepsilon^2\end{pmatrix}$ has
eigenvalues $2 + \varepsilon^2$ and $\varepsilon^2$, so the singular values are
$\sigma_1 = \sqrt{2+\varepsilon^2} \approx 1.414$ and $\sigma_2 = \varepsilon = 10^{-8}$. In
double precision, however, $1 + \varepsilon^2 = 1 + 10^{-16}$ rounds to exactly $1$, so the
computed $A^\top A$ is $\begin{pmatrix}1&1\\1&1\end{pmatrix}$ with eigenvalues $2$ and $0$ (or a
tiny negative number, whose square root is `nan`). The recipe returns $\sigma_2 = 0$ and declares
$A$ rank-deficient, although its true rank is 2 and `np.linalg.svd`, which works on $A$
directly, returns $10^{-8}$ correctly.

The mechanism is squaring: the singular values of $A^\top A$ are $\sigma_i^2$, so
$\kappa(A^\top A) = \kappa(A)^2$. Any singular value below about $\sqrt{\epsilon_{\text{mach}}}
\,\sigma_1 \approx 10^{-8}\sigma_1$ is pushed below the rounding floor $\epsilon_{\text{mach}}
\sigma_1^2 \approx 10^{-16}\sigma_1^2$ of $A^\top A$ and is lost. On random matrices all singular
values are of similar size, so the two methods agree — the difference only shows up on the
ill-conditioned matrices where accuracy matters most. The same argument says: solve least
squares via QR or SVD of $X$, never by inverting $X^\top X$.

</details>

## Exercise 12 — Mini-project: a latent-factor recommender with truncated SVD  `★★★`

Recommender systems predict the ratings a user *has not* given from the ones they have. A
ratings matrix with missing entries has no SVD, so we fill the gaps and iterate:

1. Fill every missing cell with the mean of the observed **training** ratings.
2. Repeat ~30 times: take the rank-$k$ truncated SVD of the filled matrix, then overwrite the
   *missing* cells with their rank-$k$ reconstruction while keeping the observed training
   ratings fixed (each pass is an Eckart–Young projection; the iteration converges to a
   low-rank completion).
3. Predict the held-out **test** ratings from the final rank-$k$ reconstruction and report
   the RMSE.

Run this for $k \in \{1, 2, 3, 4, 5, 6, 8, 10, 15, 20\}$, plot test RMSE (and training
RMSE) against $k$, and compare with the baseline that predicts the global training mean for
everything. The cell below generates the data: 200 users, 100 items, a true latent rank of 4,
observation noise, and 25 % of cells observed; 20 % of those are held out for testing.

**Rubric — you have mastered this when you can:**

- The imputation loop keeps observed training ratings fixed and only updates missing cells.
- Training RMSE decreases monotonically with $k$, while test RMSE is U-shaped — state at
  which $k$ it is minimal and relate that to the true rank 4.
- The best model beats the global-mean baseline by a wide margin (roughly 0.45 vs 1.04).
- The plot has a title, axis labels and a legend distinguishing train, test and baseline.
- You can explain in two sentences why large $k$ overfits even though the SVD is "optimal":
  it reproduces the noise *and the imputed values* in the observed/missing cells.
- (Bonus) Replace the global-mean fill with user-mean or item-mean fills, or add the user and
  item biases, and report whether the test RMSE improves.

In [ ]:
# ---- synthetic ratings: 200 users x 100 items, true latent rank 4, 25% observed ----
g = np.random.default_rng(0)
n_users, n_items, k_true = 200, 100, 4
P_true = g.normal(size=(n_users, k_true))
Q_true = g.normal(size=(n_items, k_true))
user_bias = g.normal(0, 0.5, size=n_users)
item_bias = g.normal(0, 0.5, size=n_items)
R_true = np.clip(3.5 + user_bias[:, None] + item_bias[None, :] + P_true @ Q_true.T / np.sqrt(k_true), 1, 5)
R_noisy = R_true + g.normal(0, 0.3, size=R_true.shape)
observed = g.random(R_true.shape) < 0.25           # which ratings exist
test_mask = observed & (g.random(R_true.shape) < 0.2)
train_mask = observed & ~test_mask
print(f"observed ratings: {observed.sum()} of {R_true.size} cells,"
      f" train {train_mask.sum()}, held-out test {test_mask.sum()}")


# YOUR CODE HERE
# def impute_svd(R, mask, k, n_iter=30): ...
# for k in [1, 2, 3, 4, 5, 6, 8, 10, 15, 20]: fit on train_mask, evaluate RMSE on test_mask
# plot RMSE vs k with the global-mean baseline as a horizontal line

## Score

In [ ]:
summary()